In [82]:
import pandas as pd
import numpy as np

departments = pd.read_csv("departments.csv")
diagnosis = pd.read_csv("diagnosis.csv")
encounters = pd.read_csv("encounters.csv")
patients = pd.read_csv("patients.csv")
providers = pd.read_csv("providers.csv")
social_determinants = pd.read_csv("social_determinants.csv")
tigercensuscodes = pd.read_csv("tigercensuscodes.csv")

/var/folders/gp/m5jhv8y904l6263shvkxvlq40000gn/T/ipykernel_7057/3376211003.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  social_determinants = pd.read_csv("social_determinants.csv")


## Null Values

In [14]:
print(departments.shape[0])
departments.isnull().sum()

11597


DepartmentKey             0
Address                2884
City                   2204
County                  574
DepartmentName            0
DepartmentSpecialty     574
DepartmentType          574
PostalCode             3009
CensusTract            4575
dtype: int64

In [13]:
print(diagnosis.shape[0])
diagnosis.isnull().sum()

1531262


DiagnosisKey      0
GroupName         0
GroupCode         0
DiagnosisName     0
DiagnosisValue    0
dtype: int64

In [12]:
print(encounters.shape[0])
encounters.isnull().sum()

7675801


Date                                 0
AdmissionInstant               6627001
AdmitYear                      6627001
AdmitMonth                     6627001
AdmitDay                       6627001
AdmitHour                      6627001
AdmitMinute                    6627001
AdmissionSource                   1400
AdmissionType                     1086
DischargeInstant               6627001
DischargeYear                  6627001
DischargeMonth                 6627001
DischargeDay                   6627001
DischargeHour                  6627001
DischargeMinute                6627001
EncounterKey                         0
PatientDurableKey                    0
Type                                 0
VisitType                            0
VisitTypeDescription                 0
ProviderDurableKey                   0
AttendingProviderDurableKey          0
DischargeProviderDurableKey          0
DepartmentKey                        0
PrimaryDiagnosisKey                  0
IsEdVisit                

In [11]:
print(patients.shape[0])
patients.isnull().sum()

947685


CensusBlockGroupFipsCode         0
DurableKey                       0
FirstRace                   365398
MaritalStatus                    0
MyChartStatus                    0
OmbEthnicity                     0
OmbRace                          0
SexAssignedAtBirth               0
SexualOrientation                0
SmokingStatus                    0
VitalStatus                      0
PatientBirthYearBin           1725
dtype: int64

In [10]:
print(providers.shape[0])
providers.isnull().sum()

299075


DurableKey                0
ClinicianTitle       113067
OfficeAddress         30335
OfficeCity            30344
OfficePostalCode      30362
PrimaryDepartment         0
PrimarySpecialty      37856
Type                      0
dtype: int64

In [39]:
supporting = pd.read_csv("Social Determinant Questions and Domains.csv")
social_determinants_supported = pd.merge(social_determinants, supporting)

print(social_determinants_supported.shape[0])
social_determinants_supported.isnull().sum()

725197


DisplayName          0
AnswerText           0
EncounterKey         0
PatientDurableKey    0
Domain               0
dtype: int64

In [16]:
print(tigercensuscodes.shape[0])
tigercensuscodes.isnull().sum()

2463


GEOID              0
PopulationValue    2
CENTLAT            2
CENTLON            2
dtype: int64

## County Population Counts

In [170]:
county_mapping = pd.DataFrame({
    "CountyCode": [
        "001","003","005","007","009","011","013","015","017","019","021","023","025","027",
        "029","031","033","035","037","039","041","043","045","047","049","051","053","055",
        "057","059","061","063","065","067","069","071","073","075","077","079","081","083",
        "085","087","089","091","093","095","097","099","101","103","105","107","109","111",
        "113","115","117","119","121","123","125","127","129","131","133","135","137","139",
        "141","143","145","147","149","151","153","155","157","159","161","163","165","167",
        "169","171","173","175","177","179","181","183","185","187","189","191","193","195",
        "197","199","201","203","205","207","209"
    ],
    "CountyName": [
        "Allen","Anderson","Atchison","Barber","Barton","Bourbon","Brown","Butler","Chase",
        "Chautauqua","Cherokee","Cheyenne","Clark","Clay","Cloud","Coffey","Comanche","Cowley",
        "Crawford","Decatur","Dickinson","Doniphan","Douglas","Edwards","Elk","Ellis","Ellsworth",
        "Finney","Ford","Franklin","Geary","Gove","Graham","Grant","Gray","Greeley","Greenwood",
        "Hamilton","Harper","Harvey","Haskell","Hodgeman","Jackson","Jefferson","Jewell",
        "Johnson","Kearny","Kingman","Kiowa","Labette","Lane","Leavenworth","Lincoln","Linn",
        "Logan","Lyon","McPherson","Marion","Marshall","Meade","Miami","Mitchell","Montgomery",
        "Morris","Morton","Nemaha","Neosho","Ness","Norton","Osage","Osborne","Ottawa","Pawnee",
        "Phillips","Pottawatomie","Pratt","Rawlins","Reno","Republic","Rice","Riley","Rooks",
        "Rush","Russell","Saline","Scott","Sedgwick","Seward","Shawnee","Sheridan","Sherman",
        "Smith","Stafford","Stanton","Stevens","Sumner","Thomas","Trego","Wabaunsee","Wallace",
        "Washington","Wichita","Wilson","Woodson","Wyandotte"
    ]
})

In [171]:
patients = patients[
    patients["CensusBlockGroupFipsCode"].notna() &
    (patients["CensusBlockGroupFipsCode"] != "*Unspecified")
] # removing patients with null or unspecified census block group information - unsure whether or not to do this 

tigercensuscodes.head()

tigercensuscodes["GEOID"] = tigercensuscodes["GEOID"].astype(str)
tigercensuscodes["CountyFips"] = tigercensuscodes["GEOID"].str[:5]


tiger_geo = tigercensuscodes[[
    "GEOID",
    "CountyFips",
    "CENTLAT",
    "CENTLON",
    "PopulationValue"
]]


patients_geo = patients.merge(
    tiger_geo,
    left_on="CensusBlockGroupFipsCode",
    right_on="GEOID",
    how="left"
)

patients_geo["CountyCode"] = patients_geo["CountyFips"].str[-3:]

In [172]:
patients_geo.head()

,CensusBlockGroupFipsCode,DurableKey,FirstRace,MaritalStatus,MyChartStatus,OmbEthnicity,OmbRace,SexAssignedAtBirth,SexualOrientation,SmokingStatus,VitalStatus,PatientBirthYearBin,GEOID,CountyFips,CENTLAT,CENTLON,PopulationValue,CountyCode
0,201770007002,5.0,White or Caucasian,Single,Activated,Not Hispanic or Latino,White,*Unspecified,*Unspecified,Never,Alive,1970.0,201770007002,20177,39.085523,-95.701250,1677.0,177
1,201770030011,15.0,White or Caucasian,Single,Activated,Not Hispanic or Latino,White,*Unspecified,*Unspecified,Every Day,Alive,1985.0,201770030011,20177,39.011938,-95.658068,1683.0,177
2,200850826001,17.0,White or Caucasian,Married,Activated,Asked but No Answer,White,*Unspecified,*Unspecified,Never,Alive,1960.0,200850826001,20085,39.306428,-95.655482,1757.0,085
3,201770016032,19.0,White or Caucasian,Single,Pending Activation,Unknown,White,*Unspecified,*Unspecified,Never Assessed,Alive,2000.0,201770016032,20177,39.008475,-95.722758,1423.0,177
4,201314802002,45.0,White or Caucasian,Married,Activated,Not Hispanic or Latino,White,*Unspecified,*Unspecified,Former,Alive,1970.0,201314802002,20131,39.844185,-96.071251,1421.0,131


In [173]:
patients_geo.groupby("CountyCode").size()

CountyCode
001     365
003     366
005    2118
007      12
009     398
       ... 
197    4150
201    1262
205     182
207     430
209     465
Length: 85, dtype: int64

## Cancer

In [80]:
df = pd.merge(encounters, diagnosis, left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey')
df['CANCER'] = df['GroupName'].str.contains('malignant', case=False, na=False).astype(int)
cancer_df = df.loc[df["CANCER"] == 1]
cancer_df["Date"] = pd.to_datetime(cancer_df["Date"], format='mixed')

/var/folders/gp/m5jhv8y904l6263shvkxvlq40000gn/T/ipykernel_7057/3029262007.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cancer_df["Date"] = pd.to_datetime(cancer_df["Date"], format='mixed')


In [102]:
# df should be a sub-dataframe of encounters with all encounters involving the relevant diagnoses
# in your input df, make sure you change the "Date" column to datetime objects
def time_between_visits(df):
    results = []
    for patient_key in df["PatientDurableKey"].unique():
        patient_df = df.loc[df["PatientDurableKey"] == patient_key].sort_values(by = "Date", ascending = True)
        
        if len(patient_df) == 1:
            avg_wait = pd.to_timedelta(0, unit = 's')
            
        else: 
            time_btwn = []
            for i in range(0, len(patient_df) - 1):
                time_btwn.append(patient_df.iloc[i+1, 0] - patient_df.iloc[i, 0])
            avg_wait = np.mean(time_btwn)

        results.append({"Patient Key": patient_key, "Avg Wait Time": avg_wait})

    return pd.DataFrame(results)

In [117]:
cancer_waittime_df = time_between_visits(cancer_df)

In [151]:
# filter out patients that only had 1 appointment
cancer_waittime_df = cancer_waittime_df.loc[cancer_waittime_df["Avg Wait Time"] != pd.Timedelta(0, unit = 's')]

In [128]:
q1 = np.percentile(cancer_waittime_df["Avg Wait Time"], 25)
q3 = np.percentile(cancer_waittime_df["Avg Wait Time"], 75)

IQR = q3 - q1

upper_fence = q3 + 1.5 * IQR
lower_fence = q1 - 1.5 * IQR

outlier_bool = (cancer_waittime_df["Avg Wait Time"] > upper_fence) | (cancer_waittime_df["Avg Wait Time"] < lower_fence)
outliers_cancer = cancer_waittime_df.loc[outlier_bool]

cancer_wt_clean = cancer_waittime_df[~ cancer_waittime_df["Patient Key"].isin(outliers_cancer["Patient Key"])]

In [192]:
cancer_county_df = pd.merge(cancer_wt_clean, patients_geo[["DurableKey", "CountyCode"]], left_on = 'Patient Key', right_on = 'DurableKey', how = 'inner')
cancer_county_df = cancer_county_df[["Patient Key", "Avg Wait Time", "CountyCode"]]


cancer_county_wt = (cancer_county_df.groupby("CountyCode").agg(AvgWaitTime = ("Avg Wait Time", "mean"), NumPatients=("Patient Key", "nunique"))
      .reset_index())
cancer_county_wt.head()

,CountyCode,AvgWaitTime,NumPatients
0,001,108 days 17:55:56.307692308,10
1,003,120 days 05:36:00,6
2,005,127 days 13:18:03.297255044,137
3,009,71 days 14:40:00,3
4,013,112 days 17:10:55.927991172,229


## Fractures

In [138]:
fractures_df = df[df["GroupName"].str.contains('fracture', case = False)]
fractures_df = fractures_df[~fractures_df["GroupName"].str.contains('osteoporosis without current pathological fracture', case = False)]
fractures_df["Date"] = pd.to_datetime(fractures_df["Date"], format='mixed')

In [139]:
fractures_waittime_df = time_between_visits(fractures_df)

In [153]:
# filter out patients that only had 1 appointment
fractures_wt = fractures_waittime_df.loc[fractures_waittime_df["Avg Wait Time"] != pd.Timedelta(0, unit = 's')]

In [157]:
q1 = np.percentile(fractures_wt["Avg Wait Time"], 25)
q3 = np.percentile(fractures_wt["Avg Wait Time"], 75)

IQR = q3 - q1

upper_fence = q3 + 1.5 * IQR
lower_fence = q1 - 1.5 * IQR

outlier_bool = (fractures_wt["Avg Wait Time"] > upper_fence) | (fractures_wt["Avg Wait Time"] < lower_fence)
outliers_fractures = fractures_wt.loc[outlier_bool]

fractures_wt_clean = fractures_wt[~fractures_wt["Patient Key"].isin(outliers_cancer["Patient Key"])]

In [191]:
fractures_county_df = pd.merge(fractures_wt_clean, patients_geo[["DurableKey", "CountyCode"]], left_on = 'Patient Key', right_on = 'DurableKey', how = 'inner')
fractures_county_df = fractures_county_df[["Patient Key", "Avg Wait Time", "CountyCode"]]


fractures_county_wt = (fractures_county_df.groupby("CountyCode").agg(AvgWaitTime = ("Avg Wait Time", "mean"), NumPatients=("Patient Key", "nunique"))
      .reset_index())
fractures_county_wt.head()

,CountyCode,AvgWaitTime,NumPatients
0,001,5 days 06:00:00,1
1,003,14 days 13:45:00,4
2,005,24 days 12:07:16.717827627,22
3,013,26 days 15:16:36.147010884,38
4,015,600 days 04:17:08.571428568,2
